# Labeldruck: Master-Target und Excel-Dateien

Dieses Colab-Notebook:

1. liest im Source-Sheet **nur die angezeigten Werte** (keine Formeln),
2. übernimmt nur Zeilen, deren Spalte **AA** mit einer Zahl und einem Bindestrich beginnt,
3. erzeugt je AA-Gruppe einen Reiter in der bestehenden Master-Target-Datei,
4. schreibt pro Source-Zeile drei Labelzeilen:
   - Equipment: EQM Nummer in **A**, Messstellen-Beschreibung in **C**
   - Functional Location: neue FLO in **A**, Messstellen-Beschreibung in **C**
   - Legacy-Bezeichnung: Asset ID / alt in **A**
5. formatiert alle geschriebenen Zellen als Text,
   wobei nur befüllte Datenzellen orange markiert werden,
6. exportiert erst nach einer ausdrücklichen **Ja/Nein-Abfrage** je Reiter eine XLSX-Datei in den Zielordner.

> **Wichtig:** Der Master-Schritt ersetzt alle vorhandenen Reiter in der konfigurierten Master-Target-Datei. Die XLSX-Dateien werden erst im letzten Abschnitt und nur nach Bestätigung geschrieben.

## 1. Setup

In [ ]:
# In Google Colab ausführen
!pip -q install "openpyxl>=3.1,<4"

from collections import OrderedDict
from copy import copy
from pathlib import Path
import io
import re
import shutil

import pandas as pd
from IPython.display import display
from google.colab import auth, files
import google.auth
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from openpyxl import Workbook, load_workbook

pd.set_option("display.max_colwidth", 120)

In [ ]:
# Konfiguration
SOURCE_SPREADSHEET_ID = "1c5ra-S3bzWLQ9JSoUuxpiIexW57YqhtVeNd_RWHfZa4"
SOURCE_SHEET_NAME = "KAU_PE MUs Summary"

MASTER_SPREADSHEET_ID = "1mxgUZApuu-KAwGl0HiQagQWpyAvF3vcpX9Zx1-04MbY"
TARGET_DRIVE_FOLDER_ID = "1RlXfLuGxKVBWwjx833UFqlfnEUNaCron"

TEMPLATE_FILENAME = "Vorlage Importdatei.xlsx"
TEMPLATE_PATH = Path("/content") / TEMPLATE_FILENAME
LOCAL_EXPORT_DIR = Path("/content/label_exports")

# Source-Spalten (0-basiert)
COL_ASSET_ID = 4       # E
COL_DESCRIPTION = 6    # G
COL_NEW_FLO = 23       # X
COL_EQM_NUMBER = 25    # Z
COL_GROUP = 26         # AA

# Erlaubt z. B. "29 - FSP Isolator mit Tunnel" oder "18 - Validatoren PKAU"
GROUP_PATTERN = re.compile(r"^\s*\d+\s*-\s*\S.*$")

print("Source:", f"https://docs.google.com/spreadsheets/d/{SOURCE_SPREADSHEET_ID}/edit")
print("Master:", f"https://docs.google.com/spreadsheets/d/{MASTER_SPREADSHEET_ID}/edit")
print("Zielordner:", f"https://drive.google.com/drive/folders/{TARGET_DRIVE_FOLDER_ID}")

In [ ]:
# Vorlage hochladen, falls sie noch nicht in der Colab-Session liegt
if not TEMPLATE_PATH.exists():
    print(f"Bitte jetzt '{TEMPLATE_FILENAME}' auswählen.")
    uploaded = files.upload()
    if TEMPLATE_FILENAME not in uploaded:
        raise FileNotFoundError(
            f"Erwartet wurde '{TEMPLATE_FILENAME}'. Hochgeladen: {list(uploaded)}"
        )
    TEMPLATE_PATH.write_bytes(uploaded[TEMPLATE_FILENAME])

template_value_wb = load_workbook(TEMPLATE_PATH, data_only=True, read_only=True)
template_value_ws = template_value_wb[template_value_wb.sheetnames[0]]
template_headers = [
    ["" if template_value_ws.cell(row=r, column=c).value is None else str(template_value_ws.cell(row=r, column=c).value)
     for c in range(1, 7)]
    for r in (1, 2)
]
template_value_wb.close()

# Format-Snapshot der Vorlage. Die Exportdateien werden kompakt neu aufgebaut,
# damit die bis Zeile 1.048.576 vorformatierte Vorlage nicht pro Datei aufgebläht wird.
template_style_wb = load_workbook(TEMPLATE_PATH, data_only=False, read_only=False)
template_style_ws = template_style_wb[template_style_wb.sheetnames[0]]
template_sheet_title = template_style_ws.title
template_freeze_panes = template_style_ws.freeze_panes
template_column_widths = {
    column: template_style_ws.column_dimensions[column].width
    for column in ("A", "B", "C", "D", "E", "F")
}
template_column_widths["A"] = max(template_column_widths["A"] or 0, 40)
template_row_heights = {
    row_number: template_style_ws.row_dimensions[row_number].height
    for row_number in (1, 2, 3)
}
template_cell_styles = {}
for row_number in (1, 2, 3):
    for column_number in range(1, 7):
        source_cell = template_style_ws.cell(row=row_number, column=column_number)
        template_cell_styles[(row_number, column_number)] = {
            "font": copy(source_cell.font),
            "fill": copy(source_cell.fill),
            "border": copy(source_cell.border),
            "alignment": copy(source_cell.alignment),
            "protection": copy(source_cell.protection),
        }
template_style_wb.close()

print("Vorlage geladen:", TEMPLATE_PATH)
display(pd.DataFrame(template_headers, index=["Kopfzeile 1", "Kopfzeile 2"]))

In [ ]:
# Google-Anmeldung
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

auth.authenticate_user()
credentials, _ = google.auth.default(scopes=SCOPES)
sheets_api = build("sheets", "v4", credentials=credentials, cache_discovery=False)
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)

print("Google-Anmeldung erfolgreich.")

## 2. Source-Werte lesen und prüfen

In [ ]:
def normalize_row(row, width=27):
    values = ["" if value is None else str(value) for value in row]
    return (values + [""] * width)[:width]


def group_sort_key(group_name):
    match = re.match(r"^\s*(\d+)", group_name)
    return (int(match.group(1)) if match else 10**9, group_name.casefold())


def safe_sheet_title(raw_title, used_titles):
    # Google-Sheets-Tabnamen: max. 100 Zeichen; []:*?/\\ sind unzulässig.
    base = re.sub(r"[\[\]:*?/\\]", "-", raw_title).strip().strip("'")
    base = base[:100] or "Ohne Name"
    candidate = base
    counter = 2
    while candidate.casefold() in used_titles:
        suffix = f" ~{counter}"
        candidate = f"{base[:100-len(suffix)]}{suffix}"
        counter += 1
    used_titles.add(candidate.casefold())
    return candidate


def safe_local_filename(raw_title):
    cleaned = re.sub(r"[\\/:*?\"<>|\x00-\x1f]", "-", raw_title).strip().rstrip(".")
    return (cleaned[:180] or "Ohne Name") + ".xlsx"


def clean_text_value(value):
    # Ein führendes Apostroph wird in Tabellen oft nur als Text-Markierung benutzt.
    # Für Barcode-/Druckerimporte darf es nicht Teil des tatsächlichen Zellwerts sein.
    text = "" if value is None else str(value).strip()
    return text.lstrip("'’﻿")


def build_label_rows(source_rows):
    label_rows = []
    for row in source_rows:
        description = clean_text_value(row[COL_DESCRIPTION])
        eqm_number = clean_text_value(row[COL_EQM_NUMBER])
        new_flo = clean_text_value(row[COL_NEW_FLO])
        asset_id = clean_text_value(row[COL_ASSET_ID])
        label_rows.extend([
            [eqm_number, "", description, "", "", ""],
            [new_flo, "", description, "", "", ""],
            [asset_id, "", "", "", "", ""],
        ])
    return label_rows


# FORMATTED_VALUE liefert die sichtbaren Ergebnisse von Formeln, nicht die Formeln selbst.
source_result = (
    sheets_api.spreadsheets()
    .values()
    .get(
        spreadsheetId=SOURCE_SPREADSHEET_ID,
        range=f"'{SOURCE_SHEET_NAME}'!A:AA",
        valueRenderOption="FORMATTED_VALUE",
        dateTimeRenderOption="FORMATTED_STRING",
        majorDimension="ROWS",
    )
    .execute()
)

all_rows = source_result.get("values", [])
if not all_rows:
    raise RuntimeError("Das Source-Sheet enthält keine lesbaren Werte.")

header = normalize_row(all_rows[0])
source_rows = [normalize_row(row) for row in all_rows[1:]]
source_row_number_by_id = {
    id(row): sheet_row_number
    for sheet_row_number, row in enumerate(source_rows, start=2)
}

expected_headers = {
    COL_ASSET_ID: "Asset ID",
    COL_DESCRIPTION: "Messstellen-Beschreibung",
    COL_NEW_FLO: "Übersetzung auf neue FLO",
    COL_EQM_NUMBER: "EQM Nummer",
}
header_errors = [
    f"{chr(65 + idx) if idx < 26 else 'AA'}: erwartet '{expected}', gefunden '{header[idx]}'"
    for idx, expected in expected_headers.items()
    if header[idx].strip() != expected
]
if header_errors:
    raise ValueError("Unerwartete Source-Kopfzeile:\n" + "\n".join(header_errors))

filtered_rows = [
    row for row in source_rows
    if GROUP_PATTERN.fullmatch(row[COL_GROUP].strip())
]

groups = OrderedDict()
for row in filtered_rows:
    groups.setdefault(row[COL_GROUP].strip(), []).append(row)
groups = OrderedDict(sorted(groups.items(), key=lambda item: group_sort_key(item[0])))

if not groups:
    raise RuntimeError("Keine gültigen Gruppen in Spalte AA gefunden.")

used_titles = set()
group_to_tab = {
    group_name: safe_sheet_title(group_name, used_titles)
    for group_name in groups
}

summary_rows = []
quality_rows = []
placeholder_pattern = re.compile(
    r"^(?:#(?:N/A|REF!|VALUE!|NAME\?|DIV/0!)|keine gefunden|nichts gefunden)$",
    re.IGNORECASE,
)

for group_name, rows in groups.items():
    summary_rows.append({
        "AA-Gruppe": group_name,
        "Master-Reiter": group_to_tab[group_name],
        "Source-Zeilen": len(rows),
        "Labelzeilen": len(rows) * 3,
        "XLSX-Dateiname": safe_local_filename(group_name),
    })
    for row in rows:
        checks = {
            "Asset ID / alt (E)": row[COL_ASSET_ID].strip(),
            "Messstellen-Beschreibung (G)": row[COL_DESCRIPTION].strip(),
            "Neue FLO (X)": row[COL_NEW_FLO].strip(),
            "EQM Nummer (Z)": row[COL_EQM_NUMBER].strip(),
        }
        for field, value in checks.items():
            if not value or placeholder_pattern.fullmatch(value):
                quality_rows.append({
                    "Source-Zeile": source_row_number_by_id[id(row)],
                    "AA-Gruppe": group_name,
                    "Feld": field,
                    "Wert": value,
                })

summary_df = pd.DataFrame(summary_rows)
quality_df = pd.DataFrame(quality_rows)

print(f"Gelesene Source-Datenzeilen: {len(source_rows):,}")
print(f"Übernommene Source-Zeilen: {len(filtered_rows):,}")
print(f"Erstellte Gruppen/Reiter: {len(groups):,}")
display(summary_df)

if quality_df.empty:
    print("Qualitätscheck: keine leeren oder offensichtlichen Fehlerwerte in E/G/X/Z.")
else:
    print(f"ACHTUNG: {len(quality_df)} auffällige Pflichtwerte. Diese werden als Text übernommen und können im Master geprüft werden.")
    display(quality_df.head(100))

In [ ]:
# Beispielprüfung für die im Auftrag genannten Source-Zeilen 5207 und 5208
def source_row_by_sheet_number(sheet_row_number):
    # all_rows enthält Kopfzeile 1 an Position 0; die Sheet-Zeilennummer passt daher zum Index + 1.
    return normalize_row(all_rows[sheet_row_number - 1])

example_rows = []
for sheet_row_number in (5207, 5208):
    row = source_row_by_sheet_number(sheet_row_number)
    example_rows.extend(
        [{"Source-Zeile": sheet_row_number, "Target A": label[0], "Target B": label[1]}
         for label in build_label_rows([row])]
    )

display(pd.DataFrame(example_rows))

## 3. Master-Target-Reiter neu erstellen

In [ ]:
def rgb(hex_color):
    value = hex_color.lstrip("#")
    return {
        "red": int(value[0:2], 16) / 255,
        "green": int(value[2:4], 16) / 255,
        "blue": int(value[4:6], 16) / 255,
    }


def get_sheet_properties(spreadsheet_id):
    metadata = (
        sheets_api.spreadsheets()
        .get(
            spreadsheetId=spreadsheet_id,
            fields="sheets.properties",
        )
        .execute()
    )
    return [sheet["properties"] for sheet in metadata.get("sheets", [])]


def write_values_in_chunks(spreadsheet_id, tab_title, values, chunk_size=3000):
    for start in range(0, len(values), chunk_size):
        chunk = values[start:start + chunk_size]
        start_row = start + 1
        end_row = start + len(chunk)
        (
            sheets_api.spreadsheets()
            .values()
            .update(
                spreadsheetId=spreadsheet_id,
                range=f"'{tab_title.replace(chr(39), chr(39)*2)}'!A{start_row}:F{end_row}",
                valueInputOption="RAW",
                body={"majorDimension": "ROWS", "values": chunk},
            )
            .execute()
        )


def recreate_master_tabs():
    existing = get_sheet_properties(MASTER_SPREADSHEET_ID)
    staging_title = "__COLAB_STAGING__"
    staging_matches = [p for p in existing if p["title"] == staging_title]

    if staging_matches:
        staging_id = staging_matches[0]["sheetId"]
    else:
        response = (
            sheets_api.spreadsheets()
            .batchUpdate(
                spreadsheetId=MASTER_SPREADSHEET_ID,
                body={"requests": [{"addSheet": {"properties": {"title": staging_title}}}]},
            )
            .execute()
        )
        staging_id = response["replies"][0]["addSheet"]["properties"]["sheetId"]

    # Master-Target ist eine dedizierte Zieldatei: alle bisherigen Reiter ersetzen.
    existing = get_sheet_properties(MASTER_SPREADSHEET_ID)
    delete_requests = [
        {"deleteSheet": {"sheetId": p["sheetId"]}}
        for p in existing
        if p["sheetId"] != staging_id
    ]
    if delete_requests:
        (
            sheets_api.spreadsheets()
            .batchUpdate(
                spreadsheetId=MASTER_SPREADSHEET_ID,
                body={"requests": delete_requests},
            )
            .execute()
        )

    add_requests = []
    for group_name, rows in groups.items():
        total_rows = 2 + len(rows) * 3
        add_requests.append({
            "addSheet": {
                "properties": {
                    "title": group_to_tab[group_name],
                    "gridProperties": {
                        "rowCount": max(100, total_rows),
                        "columnCount": 6,
                        "frozenRowCount": 2,
                    },
                }
            }
        })

    (
        sheets_api.spreadsheets()
        .batchUpdate(
            spreadsheetId=MASTER_SPREADSHEET_ID,
            body={"requests": add_requests},
        )
        .execute()
    )

    properties = get_sheet_properties(MASTER_SPREADSHEET_ID)
    title_to_id = {p["title"]: p["sheetId"] for p in properties}

    for group_name, rows in groups.items():
        tab_title = group_to_tab[group_name]
        sheet_id = title_to_id[tab_title]
        label_rows = build_label_rows(rows)
        target_values = template_headers + label_rows
        last_row = len(target_values)

        write_values_in_chunks(MASTER_SPREADSHEET_ID, tab_title, target_values)

        formatting_requests = [
            {
                "repeatCell": {
                    "range": {
                        "sheetId": sheet_id,
                        "startRowIndex": 0,
                        "endRowIndex": last_row,
                        "startColumnIndex": 0,
                        "endColumnIndex": 6,
                    },
                    "cell": {
                        "userEnteredFormat": {
                            "numberFormat": {"type": "TEXT"},
                            "textFormat": {"fontFamily": "Aptos Narrow", "fontSize": 11},
                        }
                    },
                    "fields": "userEnteredFormat(numberFormat,textFormat.fontFamily,textFormat.fontSize)",
                }
            },
            {
                "repeatCell": {
                    "range": {
                        "sheetId": sheet_id,
                        "startRowIndex": 0,
                        "endRowIndex": 2,
                        "startColumnIndex": 0,
                        "endColumnIndex": 6,
                    },
                    "cell": {
                        "userEnteredFormat": {
                            "backgroundColor": rgb("#D8D8D8"),
                            "textFormat": {
                                "bold": True,
                                "foregroundColor": rgb("#000000"),
                            },
                            "borders": {
                                edge: {
                                    "style": "SOLID",
                                    "color": rgb("#000000"),
                                }
                                for edge in ("top", "bottom", "left", "right")
                            },
                        }
                    },
                    "fields": "userEnteredFormat(backgroundColor,textFormat.bold,textFormat.foregroundColor,borders)",
                }
            },
            {
                "repeatCell": {
                    "range": {
                        "sheetId": sheet_id,
                        "startRowIndex": 1,
                        "endRowIndex": 2,
                        "startColumnIndex": 0,
                        "endColumnIndex": 6,
                    },
                    "cell": {
                        "userEnteredFormat": {
                            "wrapStrategy": "WRAP",
                            "verticalAlignment": "MIDDLE",
                        }
                    },
                    "fields": "userEnteredFormat(wrapStrategy,verticalAlignment)",
                }
            },
            {
                "addConditionalFormatRule": {
                    "rule": {
                        "ranges": [{
                            "sheetId": sheet_id,
                            "startRowIndex": 2,
                            "endRowIndex": last_row,
                            "startColumnIndex": 0,
                            "endColumnIndex": 6,
                        }],
                        "booleanRule": {
                            "condition": {
                                "type": "CUSTOM_FORMULA",
                                "values": [{"userEnteredValue": "=LEN(A3)>0"}],
                            },
                            "format": {
                                "backgroundColor": rgb("#FFC000"),
                                "textFormat": {
                                    "foregroundColor": rgb("#FF0000"),
                                },
                            },
                        },
                    },
                    "index": 0,
                }
            },
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": 0,
                        "endIndex": 1,
                    },
                    "properties": {"pixelSize": 330},
                    "fields": "pixelSize",
                }
            },
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": 1,
                        "endIndex": 2,
                    },
                    "properties": {"pixelSize": 180},
                    "fields": "pixelSize",
                }
            },
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": 2,
                        "endIndex": 3,
                    },
                    "properties": {"pixelSize": 380},
                    "fields": "pixelSize",
                }
            },
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": 3,
                        "endIndex": 5,
                    },
                    "properties": {"pixelSize": 220},
                    "fields": "pixelSize",
                }
            },
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": 5,
                        "endIndex": 6,
                    },
                    "properties": {"pixelSize": 260},
                    "fields": "pixelSize",
                }
            },
            {
                "updateDimensionProperties": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "ROWS",
                        "startIndex": 1,
                        "endIndex": 2,
                    },
                    "properties": {"pixelSize": 48},
                    "fields": "pixelSize",
                }
            },
        ]

        (
            sheets_api.spreadsheets()
            .batchUpdate(
                spreadsheetId=MASTER_SPREADSHEET_ID,
                body={"requests": formatting_requests},
            )
            .execute()
        )

    # Staging-Reiter erst löschen, nachdem alle Zielreiter sicher existieren.
    (
        sheets_api.spreadsheets()
        .batchUpdate(
            spreadsheetId=MASTER_SPREADSHEET_ID,
            body={"requests": [{"deleteSheet": {"sheetId": staging_id}}]},
        )
        .execute()
    )


recreate_master_tabs()

final_properties = get_sheet_properties(MASTER_SPREADSHEET_ID)
final_titles = [p["title"] for p in final_properties]
expected_titles = list(group_to_tab.values())
if final_titles != expected_titles:
    raise AssertionError(
        "Reiterprüfung fehlgeschlagen.\n"
        f"Erwartet: {expected_titles}\nGefunden: {final_titles}"
    )

print(f"Master-Target erfolgreich mit {len(final_titles)} Reitern erstellt.")
print(f"Öffnen: https://docs.google.com/spreadsheets/d/{MASTER_SPREADSHEET_ID}/edit")

## 4. Datenvalidierung als separaten Reiter schreiben

In [ ]:
# Dieser Check ist nicht-blockierend: Der Master bleibt bestehen und alle
# gefundenen Lücken werden im separaten Reiter 00_Datenvalidierung protokolliert.
VALIDATION_TAB_NAME = "00_Datenvalidierung"
SOURCE_SHEET_GID = 1564108399

validation_headers = [
    "Status",
    "Source-Zeile",
    "AA-Gruppe",
    "Master-Reiter",
    "Source-Feld",
    "Source-Spalte",
    "Vorhandener Wert",
    "Betroffene Labelzeile",
    "Target-Spalte",
    "Hinweis",
    "Source-Link",
]

validation_rules = [
    {
        "field": "EQM Nummer",
        "source_column": "Z",
        "index": COL_EQM_NUMBER,
        "affected_labels": "Equipment",
        "target_column": "A",
    },
    {
        "field": "Messstellen-Beschreibung",
        "source_column": "G",
        "index": COL_DESCRIPTION,
        "affected_labels": "Equipment + Functional Location",
        "target_column": "C",
    },
    {
        "field": "Übersetzung auf neue FLO",
        "source_column": "X",
        "index": COL_NEW_FLO,
        "affected_labels": "Functional Location",
        "target_column": "A",
    },
    {
        "field": "Asset ID / alt",
        "source_column": "E",
        "index": COL_ASSET_ID,
        "affected_labels": "Legacy-Bezeichnung",
        "target_column": "A",
    },
]

validation_issues = []

for group_name, rows in groups.items():
    for row in rows:
        source_row_number = source_row_number_by_id[id(row)]
        source_link = (
            f"https://docs.google.com/spreadsheets/d/{SOURCE_SPREADSHEET_ID}/edit"
            f"#gid={SOURCE_SHEET_GID}&range=A{source_row_number}:AA{source_row_number}"
        )

        for rule in validation_rules:
            raw_value = row[rule["index"]]
            clean_value = clean_text_value(raw_value)

            if not clean_value:
                status = "FEHLT"
                reason = "Source-Zelle ist leer."
            elif placeholder_pattern.fullmatch(clean_value):
                status = "PRÜFEN"
                reason = "Source enthält einen Fehler- oder Platzhalterwert."
            else:
                continue

            validation_issues.append([
                status,
                str(source_row_number),
                group_name,
                group_to_tab[group_name],
                rule["field"],
                rule["source_column"],
                clean_value,
                rule["affected_labels"],
                rule["target_column"],
                reason,
                source_link,
            ])

validation_issues.sort(key=lambda row: (int(row[1]), row[5]))

if validation_issues:
    validation_values = [validation_headers] + validation_issues
else:
    validation_values = [
        validation_headers,
        [
            "OK",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "Keine fehlenden oder offensichtlichen Fehlerwerte in E, G, X und Z gefunden.",
            "",
        ],
    ]

# Einen eventuell vorhandenen Prüf-Reiter gezielt ersetzen.
existing_properties = get_sheet_properties(MASTER_SPREADSHEET_ID)
existing_validation_tabs = [
    properties
    for properties in existing_properties
    if properties["title"] == VALIDATION_TAB_NAME
]

if existing_validation_tabs:
    (
        sheets_api.spreadsheets()
        .batchUpdate(
            spreadsheetId=MASTER_SPREADSHEET_ID,
            body={
                "requests": [
                    {
                        "deleteSheet": {
                            "sheetId": existing_validation_tabs[0]["sheetId"]
                        }
                    }
                ]
            },
        )
        .execute()
    )

add_validation_response = (
    sheets_api.spreadsheets()
    .batchUpdate(
        spreadsheetId=MASTER_SPREADSHEET_ID,
        body={
            "requests": [
                {
                    "addSheet": {
                        "properties": {
                            "title": VALIDATION_TAB_NAME,
                            "index": 0,
                            "gridProperties": {
                                "rowCount": max(100, len(validation_values)),
                                "columnCount": len(validation_headers),
                                "frozenRowCount": 1,
                            },
                        }
                    }
                }
            ]
        },
    )
    .execute()
)

validation_sheet_id = (
    add_validation_response["replies"][0]["addSheet"]["properties"]["sheetId"]
)

(
    sheets_api.spreadsheets()
    .values()
    .update(
        spreadsheetId=MASTER_SPREADSHEET_ID,
        range=f"'{VALIDATION_TAB_NAME}'!A1:K{len(validation_values)}",
        valueInputOption="RAW",
        body={
            "majorDimension": "ROWS",
            "values": validation_values,
        },
    )
    .execute()
)

validation_format_requests = [
    {
        "repeatCell": {
            "range": {
                "sheetId": validation_sheet_id,
                "startRowIndex": 0,
                "endRowIndex": len(validation_values),
                "startColumnIndex": 0,
                "endColumnIndex": len(validation_headers),
            },
            "cell": {
                "userEnteredFormat": {
                    "numberFormat": {"type": "TEXT"},
                    "verticalAlignment": "MIDDLE",
                }
            },
            "fields": "userEnteredFormat(numberFormat,verticalAlignment)",
        }
    },
    {
        "repeatCell": {
            "range": {
                "sheetId": validation_sheet_id,
                "startRowIndex": 0,
                "endRowIndex": 1,
                "startColumnIndex": 0,
                "endColumnIndex": len(validation_headers),
            },
            "cell": {
                "userEnteredFormat": {
                    "backgroundColor": rgb("#B91C1C"),
                    "textFormat": {
                        "bold": True,
                        "foregroundColor": rgb("#FFFFFF"),
                    },
                    "wrapStrategy": "WRAP",
                }
            },
            "fields": (
                "userEnteredFormat("
                "backgroundColor,textFormat.bold,"
                "textFormat.foregroundColor,wrapStrategy)"
            ),
        }
    },
    {
        "setBasicFilter": {
            "filter": {
                "range": {
                    "sheetId": validation_sheet_id,
                    "startRowIndex": 0,
                    "endRowIndex": len(validation_values),
                    "startColumnIndex": 0,
                    "endColumnIndex": len(validation_headers),
                }
            }
        }
    },
]

validation_column_widths = [
    90,   # Status
    105,  # Source-Zeile
    260,  # AA-Gruppe
    260,  # Master-Reiter
    230,  # Source-Feld
    100,  # Source-Spalte
    210,  # Vorhandener Wert
    260,  # Betroffene Labelzeile
    105,  # Target-Spalte
    300,  # Hinweis
    420,  # Source-Link
]

for column_index, pixel_size in enumerate(validation_column_widths):
    validation_format_requests.append({
        "updateDimensionProperties": {
            "range": {
                "sheetId": validation_sheet_id,
                "dimension": "COLUMNS",
                "startIndex": column_index,
                "endIndex": column_index + 1,
            },
            "properties": {"pixelSize": pixel_size},
            "fields": "pixelSize",
        }
    })

(
    sheets_api.spreadsheets()
    .batchUpdate(
        spreadsheetId=MASTER_SPREADSHEET_ID,
        body={"requests": validation_format_requests},
    )
    .execute()
)

print(
    f"Datenvalidierung abgeschlossen: {len(validation_issues)} Hinweis(e)."
)
print(
    f"https://docs.google.com/spreadsheets/d/{MASTER_SPREADSHEET_ID}/edit"
    f"#gid={validation_sheet_id}"
)

## 5. Master-Target kontrollieren

In [ ]:
# Kleine Rückleseprüfung: Kopfzeilen und erste sechs Labelzeilen je Reiter
verification_rows = []
for group_name, tab_title in group_to_tab.items():
    result = (
        sheets_api.spreadsheets()
        .values()
        .get(
            spreadsheetId=MASTER_SPREADSHEET_ID,
            range=f"'{tab_title.replace(chr(39), chr(39)*2)}'!A1:F8",
            valueRenderOption="FORMATTED_VALUE",
        )
        .execute()
    )
    values = result.get("values", [])
    verification_rows.append({
        "Reiter": tab_title,
        "gelesene Prüfzeilen": len(values),
        "erste Kopfzelle": values[0][0] if values and values[0] else "",
        "erste Labelzeile A": values[2][0] if len(values) > 2 and values[2] else "",
        "erste Labelzeile B": values[2][1] if len(values) > 2 and len(values[2]) > 1 else "",
    })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)

bad_checks = verification_df[
    (verification_df["erste Kopfzelle"] != template_headers[0][0])
    | (verification_df["gelesene Prüfzeilen"] < 5)
]
if not bad_checks.empty:
    raise AssertionError("Mindestens ein Master-Reiter hat die Rückleseprüfung nicht bestanden.")

print("Rückleseprüfung bestanden. Bitte jetzt die Master-Target-Datei visuell kontrollieren:")
print(f"https://docs.google.com/spreadsheets/d/{MASTER_SPREADSHEET_ID}/edit")

## 6. Optional: XLSX-Dateien schreiben und in Drive ablegen

In [ ]:
def apply_template_style(template_row, column_number, target_cell):
    style = template_cell_styles[(template_row, column_number)]
    target_cell.font = copy(style["font"])
    target_cell.fill = copy(style["fill"])
    target_cell.border = copy(style["border"])
    target_cell.alignment = copy(style["alignment"])
    target_cell.protection = copy(style["protection"])
    target_cell.number_format = "@"
    target_cell.quotePrefix = False


def create_xlsx_from_template(group_name, source_rows, output_path):
    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = template_sheet_title
    label_rows = build_label_rows(source_rows)
    last_row = 2 + len(label_rows)

    worksheet.freeze_panes = template_freeze_panes
    worksheet.sheet_format.defaultRowHeight = 15
    for column, width in template_column_widths.items():
        worksheet.column_dimensions[column].width = width
    for row_number, height in template_row_heights.items():
        if height is not None:
            worksheet.row_dimensions[row_number].height = height

    # Kopfzeilen als sichtbare Werte (einschliesslich des angezeigten C1-Werts),
    # nicht als Formeln.
    for row_number, header_row in enumerate(template_headers, start=1):
        for column_number, value in enumerate(header_row, start=1):
            target_cell = worksheet.cell(row=row_number, column=column_number)
            apply_template_style(row_number, column_number, target_cell)
            target_cell.value = str(value)

    # Orange/rote Musterformatierung nur auf tatsächlich befüllte Datenzellen
    # übertragen. Leere Felder bleiben ohne orange Hinterlegung.
    for row_number, label_row in enumerate(label_rows, start=3):
        if template_row_heights[3] is not None:
            worksheet.row_dimensions[row_number].height = template_row_heights[3]
        for column_number, value in enumerate(label_row, start=1):
            target_cell = worksheet.cell(row=row_number, column=column_number)
            target_cell.value = str(value)
            target_cell.number_format = "@"
            target_cell.quotePrefix = False
            if value:
                apply_template_style(3, column_number, target_cell)

    worksheet.auto_filter.ref = f"A2:F{last_row}"

    workbook.save(output_path)
    workbook.close()


def find_existing_drive_file(folder_id, filename):
    escaped_name = filename.replace("\\", "\\\\").replace("'", "\\'")
    query = (
        f"'{folder_id}' in parents and "
        f"name = '{escaped_name}' and trashed = false"
    )
    response = (
        drive_api.files()
        .list(
            q=query,
            spaces="drive",
            fields="files(id,name,webViewLink)",
            pageSize=10,
        )
        .execute()
    )
    return response.get("files", [])


def upload_or_update_xlsx(local_path, folder_id, drive_filename):
    media = MediaFileUpload(
        str(local_path),
        mimetype="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        resumable=True,
    )
    existing = find_existing_drive_file(folder_id, drive_filename)
    if existing:
        file_id = existing[0]["id"]
        result = (
            drive_api.files()
            .update(
                fileId=file_id,
                body={"name": drive_filename},
                media_body=media,
                fields="id,name,webViewLink",
            )
            .execute()
        )
        action = "aktualisiert"
    else:
        result = (
            drive_api.files()
            .create(
                body={"name": drive_filename, "parents": [folder_id]},
                media_body=media,
                fields="id,name,webViewLink",
            )
            .execute()
        )
        action = "erstellt"
    return action, result


answer = input(
    "Master-Target kontrolliert. Sollen die XLSX-Dateien jetzt geschrieben "
    "und im Drive-Zielordner abgelegt werden? [ja/NEIN]: "
).strip().casefold()

if answer not in {"ja", "j", "yes", "y"}:
    print("Kein XLSX-Export ausgeführt. Die Master-Target-Datei bleibt unverändert.")
else:
    if LOCAL_EXPORT_DIR.exists():
        shutil.rmtree(LOCAL_EXPORT_DIR)
    LOCAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    export_results = []
    for group_name, rows in groups.items():
        drive_filename = safe_local_filename(group_name)
        local_path = LOCAL_EXPORT_DIR / drive_filename
        create_xlsx_from_template(group_name, rows, local_path)
        try:
            action, uploaded_file = upload_or_update_xlsx(
                local_path,
                TARGET_DRIVE_FOLDER_ID,
                drive_filename,
            )
            status = action
            drive_link = uploaded_file.get(
                "webViewLink",
                f"https://drive.google.com/open?id={uploaded_file['id']}",
            )
        except Exception as exc:
            status = "lokal erstellt; Drive-Upload fehlgeschlagen"
            drive_link = f"{type(exc).__name__}: {exc}"
            print(f"WARNUNG bei {drive_filename}: {drive_link}")

        export_results.append({
            "AA-Gruppe": group_name,
            "Datei": drive_filename,
            "Status": status,
            "Drive-Link / Fehler": drive_link,
        })

    export_df = pd.DataFrame(export_results)
    display(export_df)
    print(f"{len(export_results)} XLSX-Dateien erfolgreich erstellt/aktualisiert.")
    print(f"Zielordner: https://drive.google.com/drive/folders/{TARGET_DRIVE_FOLDER_ID}")

    # Zusätzlich ein lokales ZIP für den Fall, dass der Drive-Download blockiert ist.
    zip_path = shutil.make_archive(
        str(LOCAL_EXPORT_DIR),
        "zip",
        root_dir=LOCAL_EXPORT_DIR,
    )
    print(f"Lokales XLSX-ZIP: {zip_path}")
    download_zip = input(
        "ZIP mit allen XLSX-Dateien aus Colab herunterladen? [ja/NEIN]: "
    ).strip().casefold()
    if download_zip in {"ja", "j", "yes", "y"}:
        files.download(zip_path)

## Checks

- Source wird mit `valueRenderOption="FORMATTED_VALUE"` gelesen: Formelergebnisse werden als sichtbare Werte übernommen.
- Der Filter akzeptiert nur AA-Werte mit führender Zahl und Bindestrich.
- Jede gültige Source-Zeile erzeugt genau drei Labelzeilen.
- Die Zuordnung ist A=EQM/neue FLO/Legacy-Bezeichnung und C=Beschreibung.
- Nur befüllte Datenzellen erhalten die orange/rote Vorlagenformatierung.
- Führende Apostrophe werden aus Druckwerten entfernt; Excel erzwingt Text über das Format `@` bei deaktiviertem `quotePrefix`.
- Alle Schreibvorgänge verwenden Strings/RAW und Textformat.
- Der Reiter `00_Datenvalidierung` protokolliert fehlende bzw. offensichtliche Fehlerwerte aus E, G, X und Z, ohne den Lauf abzubrechen.
- Die Master-Reiter werden nach der numerischen Präfixzahl sortiert.
- XLSX-Dateien werden nur nach ausdrücklicher Bestätigung exportiert; vorhandene gleichnamige Dateien im Zielordner werden aktualisiert.